In [ ]:
%run ../langchain_common.py

# Multi-Agent Event Planner

In [ ]:
@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Create State

In [ ]:
from langchain.agents import AgentState

class EventState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Create Subagents


In [ ]:
# Venue agent
venue_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 5 web searches. Count every web_search call you make.
    After 5 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

In [ ]:
# Playlist agent
playlist_agent = create_agent(
    model=llm,
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

## Main Coordinator


In [ ]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> Command:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    
    # Command applies an atomic state update in LangGraph so later tool calls can use these values.
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


### Using threadid w/ checkpointers for reliable conversation!

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Create a checkpointer to persist state automatically
memory_saver = MemorySaver()

# Recreate coordinator with checkpointer
coordinator_with_memory = create_agent(
    model=llm,
    tools=[search_venues, suggest_playlist, update_state],
    state_schema=EventState,
    checkpointer=memory_saver,  # Enable automatic state persistence
    system_prompt="""
    You are a event planner & coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)

In [ ]:
def send_message_with_checkpointer(msg: str, thread_id: str):
    """
    Send a message using thread-based state management.
    State (including message history) is automatically persisted and retrieved
    via thread_id - no manual history passing needed!
    """
    response = coordinator_with_memory.invoke(
        {
            "messages": [HumanMessage(content=msg)]
        },
        config={
            "configurable": {"thread_id": thread_id},
            "tags": ["EventPlanner"],
            "recursion_limit": 40
        },
    )
    
    updated_messages = response["messages"]
    pp(updated_messages[-1].content)
    return updated_messages

In [ ]:
thread_id = new_conversation_id()
msg_list = send_message_with_checkpointer("I'm from Lahore and I'd like a wedding in Gujranwala for 100 guests. Also, create a playlist with jazz genre. Make sure that playlist has multiple artists.", thread_id)

In [ ]:
# Follow-up: State (genre, destination, guest_count, origin) is automatically preserved!
msg_list = send_message_with_checkpointer("Can you please display the playlist?", thread_id)

In [ ]:
msg_list = send_message_with_checkpointer("Cut the list in half", thread_id)

In [ ]:
msg_list = send_message_with_checkpointer("Provide the track names please as well", thread_id)

In [ ]:
msg_list = send_message_with_checkpointer("Yes, please provide the track names as well", thread_id)

In [ ]:
msg_list = send_message_with_checkpointer("provide me detail list of tracks", thread_id)

In [ ]:
msg_list = send_message_with_checkpointer("Yes", thread_id)